# 02F — Dynamic Recommendation Engine (Final Phase 2)

**Builds on:** `01_core_scoring_engine.ipynb` (Phase 1), `02_dynamic_recommendation_layer.ipynb` (Phase 2 Draft)

**Architecture:** Two-Stage NLP Pipeline — TF-IDF Cosine Filter → KNN Profile Re-ranker

---

### Pipeline Overview

```
User Parameters: anchor_region, group_size, tourist_type
        │
        ▼
   ┌────────────────────────────────────────┐
   │  STAGE 1: TF-IDF + Cosine Similarity   │
   │  Tags → TfidfVectorizer → cosine_sim   │
   │  Hard filter: score == 0 → eliminated  │
   └───────────────┬────────────────────────┘
                   │ survivors only
                   ▼
   ┌────────────────────────────────────────┐
   │  STAGE 2: KNN Profile Re-ranker        │
   │  Numerical features → NearestNeighbors │
   │  Distance → knn_profile_score (0-100)  │
   └───────────────┬────────────────────────┘
                   │
                   ▼
   base_score = (tfidf×0.50) + (knn×0.30) + (attractiveness×0.20)
   final_match_score = base_score - (congestion×0.5) - penalty
```


In [1]:
import pandas as pd
import numpy as np
import math

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler

import ipywidgets as widgets
from IPython.display import display, clear_output

print("All imports loaded successfully.")


All imports loaded successfully.


---
## Step 1 — 30-Location Japan Tourism Dataset (Expanded Tags)

Each location carries **10–15 detailed tags** spanning three conceptual tiers:

| Tier | Purpose | Examples |
|---|---|---|
| **Core Motivators** | Primary draw | `temple`, `hot_spring`, `urban` |
| **Vibe** | Atmospheric character | `serene`, `lively`, `rustic` |
| **Micro-features** | Specific activities/assets | `matcha`, `sake_brewery`, `cable_car` |

Tags are stored as **lists** for `TfidfVectorizer` compatibility.

In [2]:
# ── 30-Location Japan Tourism Dataset (Expanded Tags + Coordinates) ──────────
data = [
    # ═══════════════════════  MAJOR ANCHORS  ═══════════════════════
    {"region": "Tokyo", "is_anchor": True, "tourist_count": 200000, "capacity": 150000,
     "avg_delay_minutes": 25, "cultural_score": 85, "experiential_score": 95,
     "infrastructure_score": 95, "promotion_score": 100, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.6895, "lon": 139.6917,
     "tags": ["urban", "modern", "shopping", "food", "nightlife", "anime",
             "tech", "skyscraper", "street_food", "museum", "lively",
             "fashion", "train_hub", "neon", "pop_culture"]},

    {"region": "Kyoto", "is_anchor": True, "tourist_count": 120000, "capacity": 80000,
     "avg_delay_minutes": 30, "cultural_score": 95, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 95, "cleanliness_score": 72,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.0116, "lon": 135.7681,
     "tags": ["historic", "temple", "shrine", "cherry_blossom", "traditional",
             "unesco", "culture", "geisha", "gardens", "bamboo",
             "matcha", "photo_spot", "architecture", "autumn_leaves", "crafts"]},

    {"region": "Osaka", "is_anchor": True, "tourist_count": 150000, "capacity": 120000,
     "avg_delay_minutes": 20, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 90, "promotion_score": 90, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.6937, "lon": 135.5023,
     "tags": ["urban", "food", "nightlife", "modern", "street_food",
             "castle", "lively", "comedy", "shopping", "takoyaki",
             "neon", "canal", "entertainment"]},

    {"region": "Hakone", "is_anchor": True, "tourist_count": 60000, "capacity": 40000,
     "avg_delay_minutes": 35, "cultural_score": 85, "experiential_score": 90,
     "infrastructure_score": 80, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 35.2326, "lon": 139.1070,
     "tags": ["hot_spring", "nature", "mountain", "view", "lake",
             "ryokan", "cable_car", "volcanic", "serene", "art_museum",
             "scenic_train", "relaxation", "fuji_view"]},

    {"region": "Nara", "is_anchor": True, "tourist_count": 70000, "capacity": 50000,
     "avg_delay_minutes": 25, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 80, "promotion_score": 80, "cleanliness_score": 74,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 34.6851, "lon": 135.8048,
     "tags": ["historic", "temple", "nature", "deer", "unesco",
             "traditional", "park", "buddha", "ancient", "serene",
             "photo_spot", "culture", "architecture"]},

    {"region": "Hiroshima", "is_anchor": True, "tourist_count": 90000, "capacity": 70000,
     "avg_delay_minutes": 20, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 75, "promotion_score": 85, "cleanliness_score": 78,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.3853, "lon": 132.4553,
     "tags": ["historic", "memorial", "peace", "temple", "island",
             "unesco", "culture", "coastal", "tram", "okonomiyaki",
             "architecture", "museum", "solemn"]},

    {"region": "Kamakura", "is_anchor": True, "tourist_count": 55000, "capacity": 40000,
     "avg_delay_minutes": 28, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 80, "cleanliness_score": 75,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 35.3192, "lon": 139.5467,
     "tags": ["historic", "temple", "coastal", "buddha", "shrine",
             "hiking", "beach", "traditional", "serene", "photo_spot",
             "architecture", "crafts", "day_trip"]},

    {"region": "Nikko", "is_anchor": True, "tourist_count": 45000, "capacity": 35000,
     "avg_delay_minutes": 30, "cultural_score": 92, "experiential_score": 88,
     "infrastructure_score": 70, "promotion_score": 75, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 36.7500, "lon": 139.6000,
     "tags": ["historic", "shrine", "mountain", "nature", "unesco",
             "waterfall", "autumn_leaves", "ornate", "cedar", "lake",
             "hiking", "traditional", "architecture"]},

    {"region": "Sapporo", "is_anchor": True, "tourist_count": 80000, "capacity": 70000,
     "avg_delay_minutes": 15, "cultural_score": 75, "experiential_score": 90,
     "infrastructure_score": 85, "promotion_score": 85, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 43.0621, "lon": 141.3544,
     "tags": ["urban", "snow", "food", "festival", "beer",
             "ski", "ramen", "park", "modern", "winter_sport",
             "lively", "night_market", "scenic"]},

    {"region": "Fukuoka", "is_anchor": True, "tourist_count": 65000, "capacity": 60000,
     "avg_delay_minutes": 10, "cultural_score": 80, "experiential_score": 90,
     "infrastructure_score": 88, "promotion_score": 80, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.5904, "lon": 130.4017,
     "tags": ["urban", "food", "historic", "shopping", "ramen",
             "shrine", "night_market", "canal", "modern", "lively",
             "street_food", "port"]},

    # ═══════════════════════  ORBIT DESTINATIONS  ═══════════════════════
    {"region": "Kanazawa", "is_anchor": False, "tourist_count": 25000, "capacity": 50000,
     "avg_delay_minutes": 8, "cultural_score": 85, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.5613, "lon": 136.6562,
     "tags": ["historic", "cherry_blossom", "garden", "traditional", "crafts",
             "geisha", "samurai", "museum", "culture", "serene",
             "architecture", "matcha", "autumn_leaves"]},

    {"region": "Takayama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 80, "experiential_score": 75,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 90,
     "has_coach_parking": False, "group_dining_nearby": True, "requires_long_walk": True,
     "lat": 36.1461, "lon": 137.2517,
     "tags": ["alpine", "mountain", "rural", "historic", "traditional",
             "festival", "sake_brewery", "edo_town", "crafts", "serene",
             "hiking", "wood_carving", "morning_market"]},

    {"region": "Shirakawa-go", "is_anchor": False, "tourist_count": 22000, "capacity": 20000,
     "avg_delay_minutes": 15, "cultural_score": 90, "experiential_score": 85,
     "infrastructure_score": 50, "promotion_score": 70, "cleanliness_score": 80,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 36.2578, "lon": 136.9061,
     "tags": ["rural", "historic", "snow", "mountain", "unesco",
             "traditional", "thatched_roof", "village", "photo_spot", "serene",
             "farming", "culture", "winter"]},

    {"region": "Koyasan", "is_anchor": False, "tourist_count": 12000, "capacity": 25000,
     "avg_delay_minutes": 5, "cultural_score": 95, "experiential_score": 85,
     "infrastructure_score": 60, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 34.2130, "lon": 135.5830,
     "tags": ["temple", "historic", "mountain", "rural", "unesco",
             "buddhist", "cemetery", "serene", "meditation", "cable_car",
             "traditional", "spiritual", "pilgrim", "shojin_ryori"]},

    {"region": "Nagasaki", "is_anchor": False, "tourist_count": 20000, "capacity": 45000,
     "avg_delay_minutes": 8, "cultural_score": 88, "experiential_score": 85,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 82,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 32.7503, "lon": 129.8779,
     "tags": ["historic", "memorial", "coastal", "food", "tram",
             "culture", "church", "port", "island", "museum",
             "solemn", "international", "night_view"]},

    {"region": "Tottori", "is_anchor": False, "tourist_count": 12000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 70, "experiential_score": 78,
     "infrastructure_score": 60, "promotion_score": 40, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 35.5011, "lon": 134.2351,
     "tags": ["rural", "nature", "sand_dunes", "coastal", "adventure",
             "camel_ride", "paragliding", "museum", "quiet", "scenic",
             "seafood", "off_beaten_path"]},

    {"region": "Akita", "is_anchor": False, "tourist_count": 9000, "capacity": 30000,
     "avg_delay_minutes": 3, "cultural_score": 65, "experiential_score": 72,
     "infrastructure_score": 55, "promotion_score": 35, "cleanliness_score": 82,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 39.7186, "lon": 140.1024,
     "tags": ["rural", "nature", "festival", "mountain", "snow",
             "onsen", "lake", "samurai", "rice_paddy", "quiet",
             "traditional", "rustic"]},

    {"region": "Shimane", "is_anchor": False, "tourist_count": 7000, "capacity": 25000,
     "avg_delay_minutes": 2, "cultural_score": 75, "experiential_score": 70,
     "infrastructure_score": 50, "promotion_score": 30, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 35.4723, "lon": 133.0505,
     "tags": ["historic", "shrine", "garden", "rural", "mythology",
             "traditional", "serene", "castle", "quiet", "culture",
             "off_beaten_path", "sunset"]},

    {"region": "Okayama", "is_anchor": False, "tourist_count": 15000, "capacity": 40000,
     "avg_delay_minutes": 5, "cultural_score": 82, "experiential_score": 75,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.6551, "lon": 133.9195,
     "tags": ["historic", "garden", "castle", "culture", "traditional",
             "crafts", "denim", "fruit", "cycling", "serene",
             "architecture", "museum"]},

    {"region": "Kagawa", "is_anchor": False, "tourist_count": 14000, "capacity": 35000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 82,
     "infrastructure_score": 65, "promotion_score": 55, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 34.3401, "lon": 134.0434,
     "tags": ["food", "coastal", "shrine", "udon", "island",
             "art", "garden", "pilgrimage", "scenic", "cycling",
             "bridge", "traditional"]},

    {"region": "Naoshima", "is_anchor": False, "tourist_count": 11000, "capacity": 15000,
     "avg_delay_minutes": 12, "cultural_score": 88, "experiential_score": 90,
     "infrastructure_score": 55, "promotion_score": 60, "cleanliness_score": 88,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 34.4614, "lon": 133.9958,
     "tags": ["art", "coastal", "modern", "rural", "museum",
             "architecture", "island", "sculpture", "gallery", "photo_spot",
             "minimalist", "design", "cycling"]},

    {"region": "Matsuyama", "is_anchor": False, "tourist_count": 18000, "capacity": 40000,
     "avg_delay_minutes": 6, "cultural_score": 85, "experiential_score": 82,
     "infrastructure_score": 72, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.8392, "lon": 132.7657,
     "tags": ["hot_spring", "historic", "castle", "tram", "traditional",
             "literature", "garden", "relaxation", "food", "serene",
             "architecture", "culture"]},

    {"region": "Beppu", "is_anchor": False, "tourist_count": 25000, "capacity": 45000,
     "avg_delay_minutes": 10, "cultural_score": 75, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 65, "cleanliness_score": 80,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 33.2846, "lon": 131.4914,
     "tags": ["hot_spring", "nature", "coastal", "volcanic", "steam",
             "relaxation", "ryokan", "sand_bath", "scenic", "food",
             "hell_tour", "unique"]},

    {"region": "Kurokawa Onsen", "is_anchor": False, "tourist_count": 8000, "capacity": 12000,
     "avg_delay_minutes": 4, "cultural_score": 80, "experiential_score": 95,
     "infrastructure_score": 45, "promotion_score": 40, "cleanliness_score": 92,
     "has_coach_parking": False, "group_dining_nearby": False, "requires_long_walk": True,
     "lat": 33.1283, "lon": 131.0714,
     "tags": ["hot_spring", "rural", "mountain", "ryokan", "serene",
             "traditional", "nature", "relaxation", "lantern", "rustic",
             "intimate", "off_beaten_path"]},

    {"region": "Kagoshima", "is_anchor": False, "tourist_count": 16000, "capacity": 38000,
     "avg_delay_minutes": 5, "cultural_score": 78, "experiential_score": 85,
     "infrastructure_score": 70, "promotion_score": 50, "cleanliness_score": 83,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 31.5966, "lon": 130.5571,
     "tags": ["nature", "coastal", "mountain", "hot_spring", "volcanic",
             "scenic", "food", "garden", "historic", "tram",
             "island", "adventure", "subtropical"]},

    {"region": "Sendai", "is_anchor": False, "tourist_count": 30000, "capacity": 65000,
     "avg_delay_minutes": 8, "cultural_score": 82, "experiential_score": 80,
     "infrastructure_score": 85, "promotion_score": 60, "cleanliness_score": 84,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 38.2682, "lon": 140.8694,
     "tags": ["urban", "historic", "food", "festival", "castle",
             "shopping", "beef_tongue", "tanabata", "park", "modern",
             "day_trip", "lively"]},

    {"region": "Aomori", "is_anchor": False, "tourist_count": 10000, "capacity": 35000,
     "avg_delay_minutes": 3, "cultural_score": 75, "experiential_score": 80,
     "infrastructure_score": 60, "promotion_score": 45, "cleanliness_score": 86,
     "has_coach_parking": True, "group_dining_nearby": False, "requires_long_walk": False,
     "lat": 40.8244, "lon": 140.7400,
     "tags": ["nature", "snow", "rural", "festival", "nebuta",
             "apple", "hiking", "mountain", "lake", "scenic",
             "autumn_leaves", "quiet", "rustic"]},

    {"region": "Nagano", "is_anchor": False, "tourist_count": 22000, "capacity": 50000,
     "avg_delay_minutes": 7, "cultural_score": 85, "experiential_score": 88,
     "infrastructure_score": 75, "promotion_score": 60, "cleanliness_score": 87,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.6513, "lon": 138.1810,
     "tags": ["mountain", "snow", "nature", "temple", "ski",
             "monkey_park", "hiking", "alpine", "traditional", "soba",
             "scenic", "onsen", "winter_sport"]},

    {"region": "Matsumoto", "is_anchor": False, "tourist_count": 15000, "capacity": 35000,
     "avg_delay_minutes": 5, "cultural_score": 88, "experiential_score": 80,
     "infrastructure_score": 70, "promotion_score": 55, "cleanliness_score": 88,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.2380, "lon": 137.9720,
     "tags": ["castle", "historic", "alpine", "traditional", "culture",
             "crafts", "soba", "museum", "architecture", "serene",
             "mountain", "wasabi", "art"]},

    {"region": "Toyama", "is_anchor": False, "tourist_count": 13000, "capacity": 40000,
     "avg_delay_minutes": 4, "cultural_score": 75, "experiential_score": 85,
     "infrastructure_score": 68, "promotion_score": 50, "cleanliness_score": 85,
     "has_coach_parking": True, "group_dining_nearby": True, "requires_long_walk": False,
     "lat": 36.6953, "lon": 137.2114,
     "tags": ["alpine", "nature", "coastal", "snow", "scenic",
             "glass_art", "sushi", "firefly_squid", "mountain", "dam",
             "hiking", "modern", "quiet"]},
]

df = pd.DataFrame(data)

# Quick validation
tag_counts = df["tags"].apply(len)
print(f"Locations: {len(df)}  |  Anchors: {df['is_anchor'].sum()}  |  Orbits: {(~df['is_anchor']).sum()}")
print(f"Tags per location - min: {tag_counts.min()}, max: {tag_counts.max()}, mean: {tag_counts.mean():.1f}")
print(f"Coordinates present: {df['lat'].notna().all() and df['lon'].notna().all()}")
df[["region", "is_anchor", "lat", "lon", "cultural_score"]].head(5)


Locations: 30  |  Anchors: 10  |  Orbits: 20
Tags per location - min: 12, max: 15, mean: 12.8
Coordinates present: True


,region,is_anchor,lat,lon,cultural_score
0,Tokyo,True,35.6895,139.6917,85
1,Kyoto,True,35.0116,135.7681,95
2,Osaka,True,34.6937,135.5023,80
3,Hakone,True,35.2326,139.1070,85
4,Nara,True,34.6851,135.8048,95


---
## Step 2 — Phase 1 Helpers + Phase 3 Spatial Functions

**Phase 1 Congestion Helpers** — reproduced for self-contained execution:
- Piecewise load score, Dominance Rule, dynamic feasibility penalties.

**Phase 3 Spatial Functions** — Dynamic Transport Profile:
- **Tortuosity Multiplier (1.4×):** Converts Haversine great-circle distance to realistic route distance.
- **Dynamic Speed:** All groups use distance-based tiers — Local (40 km/h <50 km), Express (80 km/h 50–150 km), Shinkansen (150 km/h >150 km). Groups ≥ 15 apply a 1.25× logistics overhead on commute time rather than being locked to coach speed.
- **Softened Penalty:** Scales gently to max 15 pts at catchment boundary, with 1.5× friction for coaches.

In [3]:
# ── Phase 1 Congestion Helpers ────────────────────────────────────────────────
def piecewise_load_score(ratio: float) -> float:
    """Piecewise congestion score with no ceiling above ratio >= 1.2."""
    if ratio < 0.3:
        return ratio * (20 / 0.3)
    elif ratio < 0.7:
        return 20 + (ratio - 0.3) * (30 / 0.4)
    elif ratio < 0.9:
        return 50 + (ratio - 0.7) * (25 / 0.2)
    elif ratio < 1.0:
        return 75 + (ratio - 0.9) * (15 / 0.1)
    elif ratio < 1.2:
        return 90 + (ratio - 1.0) * (10 / 0.2)
    else:
        return 100 + ((ratio - 1.2) * 20)


def compute_congestion_index(load_score: float, delay_score: float) -> float:
    """Dominance Rule: delay amplifies when load_score >= 100."""
    if load_score >= 100:
        return load_score + (delay_score * 0.20)
    return (load_score * 0.70) + (delay_score * 0.30)


def congestion_flag(index: float) -> str:
    if index < 20:   return "Severely undertouristed"
    elif index < 40: return "Undertouristed"
    elif index < 75: return "Balanced"
    elif index < 100: return "Approaching overtourism"
    else:             return "Severely overtouristed"


def compute_dynamic_penalty(row: pd.Series, group_size: int) -> int:
    """Group-size-gated feasibility penalty."""
    if group_size >= 15:
        return (
            (0 if row["has_coach_parking"]  else 20) +
            (0 if row["group_dining_nearby"] else 15) +
            (10 if row["requires_long_walk"] else 0)
        )
    else:
        return (5 if row["requires_long_walk"] else 0)


# ── Phase 3 Spatial Functions (Dynamic Transport Profile) ─────────────────────
TORTUOSITY = 1.4      # Haversine → realistic route distance multiplier
COACH_OVERHEAD = 1.25  # Logistics overhead for groups >= 15 (boarding, coordination, rest stops)


def _haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in km using the Haversine formula."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


def calculate_dynamic_commute(
    lat1: float, lon1: float,
    lat2: float, lon2: float,
    group_size: int,
) -> tuple:
    """
    Dynamic commute using marginal travel-time brackets (no discontinuities).

    Public Transit brackets (cumulative, like tax brackets):
      0-50 km  : 1.50 min/km + 10 min base transfer  ->  max  85 min at 50 km
      50-150 km: 0.75 min/km (Limited Express)        ->  max 160 min at 150 km
      >150 km  : 0.40 min/km (Shinkansen)

    Returns
    -------
    (commute_minutes, transport_mode, route_distance_km)
    """
    gc_dist = _haversine_km(lat1, lon1, lat2, lon2)
    route_distance = gc_dist * TORTUOSITY

    if group_size >= 15:
        commute_minutes = (route_distance / 65.0) * 60.0
        transport_mode = "Private Coach"
    else:
        if route_distance <= 50:
            commute_minutes = (route_distance * 1.5) + 10
            transport_mode = "Local Transit"
        elif route_distance <= 150:
            commute_minutes = 85 + ((route_distance - 50) * 0.75)
            transport_mode = "Limited Express"
        else:
            commute_minutes = 160 + ((route_distance - 150) * 0.4)
            transport_mode = "Shinkansen"

    return (commute_minutes, transport_mode, round(route_distance, 1))


def calculate_distance_penalty(
    commute_minutes: float,
    group_size: int,
    max_commute_mins: float,
) -> float:
    """
    Softened distance penalty with geographic hard filter.

    - Beyond max_commute_mins → inf (hard filter).
    - Within limit → base penalty scales gently to max 15 pts at cutoff.
    - Groups >= 15 incur a 1.5× logistical friction multiplier.
    """
    if commute_minutes > max_commute_mins:
        return float('inf')
    base_penalty = (commute_minutes / max_commute_mins) * 15.0
    if group_size >= 15:
        return round(base_penalty * 1.5, 2)
    return round(base_penalty, 2)


print("Phase 1 congestion helpers + Phase 3 spatial functions loaded.")


Phase 1 congestion helpers + Phase 3 spatial functions loaded.


---
## Step 3 — Two-Stage NLP Pipeline + Dynamic Spatial Layer

### Stage 1: TF-IDF + Cosine Similarity (Thematic Hard Filter)

Each location's tag list is joined into a single string and passed through `TfidfVectorizer`.
TF-IDF up-weights **rare, distinctive tags** (`geisha`, `firefly_squid`) and down-weights
**ubiquitous tags** (`historic`, `nature`). Cosine similarity produces a 0–100 `tfidf_score`.
**Orbits scoring 0.0 are hard-filtered out.**

### Stage 2: KNN Profile Re-ranker

Surviving orbits are evaluated by `NearestNeighbors` on their **numerical quality profile**.
Cosine distance → 0–100 `knn_profile_score`.

### Phase 3: Dynamic Spatial Layer

- **Tortuosity (1.4×):** Converts Haversine distance to realistic route distance.
- **Dynamic Transport Mode:** Coach groups get 65 km/h; FIT groups get Local (40), Express (80), or Shinkansen (150) based on distance.
- **Softened Penalty:** Scales to max 15 pts at catchment boundary, with 1.5× friction for coaches.
- **Hard Filter:** Destinations beyond `max_commute_mins` are eliminated.

### Final Score (Additive Weighted Sum Model)

$$\text{base\_score} = (\text{tfidf\_score} \times 0.50) + (\text{knn\_profile\_score} \times 0.30) + (\text{attractiveness} \times 0.20)$$

$$\text{final\_match\_score} = \text{base\_score} - (\text{congestion\_index} \times 0.5) - \text{dynamic\_penalty} - \text{distance\_penalty}$$

In [4]:
# ── Two-Stage NLP + Dynamic Spatial Recommendation Engine ─────────────────────

def _compute_tfidf_scores(df: pd.DataFrame, anchor_name: str) -> dict:
    """Stage 1: TF-IDF cosine similarity (anchor vs all). Returns {region: score}."""
    tag_strings = df["tags"].apply(lambda t: " ".join(t))
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(tag_strings)
    anchor_idx = df.index[df["region"] == anchor_name][0]
    cos_sim = cosine_similarity(tfidf_matrix[anchor_idx], tfidf_matrix).flatten()
    return dict(zip(df["region"], np.round(cos_sim * 100, 2)))


def _compute_knn_profile_scores(df: pd.DataFrame, anchor_name: str,
                                 survivor_regions: list) -> dict:
    """Stage 2: KNN cosine-distance similarity on numerical features.
    Only scores survivors from Stage 1. Returns {region: score}."""
    num_cols = ["cultural_score", "experiential_score",
               "cleanliness_score", "infrastructure_score"]
    scaler = MinMaxScaler()
    num_matrix = scaler.fit_transform(df[num_cols].values)

    knn = NearestNeighbors(n_neighbors=len(df), metric="cosine")
    knn.fit(num_matrix)

    anchor_idx = df.index[df["region"] == anchor_name][0]
    distances, indices = knn.kneighbors(num_matrix[anchor_idx].reshape(1, -1))

    sim_map = {}
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        region = df.iloc[idx]["region"]
        if region in survivor_regions:
            sim_map[region] = round((1 - dist) * 100, 2)
    return sim_map


# Column schema for the output DataFrame
_OUTPUT_COLS = [
    "region", "tfidf_score", "knn_profile_score",
    "congestion_index", "congestion_flag",
    "dynamic_penalty", "commute_minutes",
    "transport_mode", "distance_penalty",
    "final_match_score",
]


def generate_recommendations(
    df: pd.DataFrame,
    anchor_region: str,
    group_size: int,
    tourist_type: str,
    max_commute_mins: int = 180,
) -> pd.DataFrame:
    """
    Two-Stage NLP + Dynamic Spatial Recommendation Engine.

    Parameters
    ----------
    df              : DataFrame with tags, scores, lat/lon.
    anchor_region   : Selected anchor destination.
    group_size      : Travel group size.
    tourist_type    : 'international' or 'native'.
    max_commute_mins: Geographic catchment cutoff in minutes (default 180).

    Returns
    -------
    DataFrame sorted by final_match_score DESC.
    """
    tourist_type = tourist_type.lower().strip()
    if tourist_type not in ("international", "native"):
        raise ValueError("tourist_type must be 'international' or 'native'")

    anchor_row = df.loc[df["region"] == anchor_region]
    if anchor_row.empty:
        raise ValueError(f"Anchor region '{anchor_region}' not found.")

    anchor_lat = anchor_row["lat"].values[0]
    anchor_lon = anchor_row["lon"].values[0]

    # ── Stage 1: TF-IDF cosine similarity ─────────────────────────────────
    tfidf_scores = _compute_tfidf_scores(df, anchor_region)

    orbits = df[df["region"] != anchor_region].copy()
    orbits["tfidf_score"] = orbits["region"].map(tfidf_scores)
    survivors = orbits[orbits["tfidf_score"] > 0.0].copy()

    if survivors.empty:
        return pd.DataFrame(columns=_OUTPUT_COLS)

    # ── Stage 2: KNN profile re-ranking ───────────────────────────────────
    survivor_list = survivors["region"].tolist()
    knn_scores = _compute_knn_profile_scores(df, anchor_region, survivor_list)
    survivors["knn_profile_score"] = survivors["region"].map(knn_scores)

    # ── Score each survivor ───────────────────────────────────────────────
    results = []
    for _, row in survivors.iterrows():

        # ── Phase 3: Dynamic commute & penalty ────────────────────────────
        commute, transport_mode, route_dist = calculate_dynamic_commute(
            anchor_lat, anchor_lon, row["lat"], row["lon"], group_size
        )
        dist_penalty = calculate_distance_penalty(
            commute, group_size, max_commute_mins
        )
        if dist_penalty == float('inf'):
            continue  # Outside catchment

        # ── Attractiveness weights ────────────────────────────────────────
        if tourist_type == "international":
            attractiveness = (
                row["cultural_score"]      * 0.40 +
                row["cleanliness_score"]   * 0.30 +
                row["experiential_score"]  * 0.20 +
                row["infrastructure_score"]* 0.10
            )
        else:
            attractiveness = (
                row["experiential_score"]  * 0.40 +
                row["infrastructure_score"]* 0.30 +
                row["cleanliness_score"]   * 0.20 +
                row["cultural_score"]      * 0.10
            )

        # ── Base score ────────────────────────────────────────────────────
        base_score = (
            row["tfidf_score"]        * 0.50 +
            row["knn_profile_score"]  * 0.30 +
            attractiveness            * 0.20
        )

        # ── Congestion ────────────────────────────────────────────────────
        load_ratio  = row["tourist_count"] / row["capacity"]
        load_score  = piecewise_load_score(load_ratio)
        delay_score = min((row["avg_delay_minutes"] / 45) * 100, 100)
        c_index     = compute_congestion_index(load_score, delay_score)
        c_flag      = congestion_flag(c_index)

        # ── Feasibility penalty ───────────────────────────────────────────
        d_penalty = compute_dynamic_penalty(row, group_size)

        # ── Final score ──────────────────────────────────────────────────
        final = round(
            base_score - (c_index * 0.5) - d_penalty - dist_penalty, 2
        )

        results.append({
            "region":             row["region"],
            "tfidf_score":        row["tfidf_score"],
            "knn_profile_score":  row["knn_profile_score"],
            "congestion_index":   round(c_index, 2),
            "congestion_flag":    c_flag,
            "dynamic_penalty":    d_penalty,
            "commute_minutes":    round(commute),
            "transport_mode":     transport_mode,
            "distance_penalty":   round(dist_penalty, 2),
            "final_match_score":  final,
        })

    if not results:
        return pd.DataFrame(columns=_OUTPUT_COLS)

    return (
        pd.DataFrame(results)
        .sort_values("final_match_score", ascending=False)
        .reset_index(drop=True)
    )


print("Two-Stage NLP + Dynamic Spatial Engine loaded.")


Two-Stage NLP + Dynamic Spatial Engine loaded.


---
## Step 4 — Scenario Execution: Kyoto (International, Coach Tour, 120-min Catchment)

Quick validation with spatial filtering. Destinations beyond 120 minutes of Kyoto are eliminated.


In [5]:
# ── Scenario: Kyoto anchor, coach tour, international, 180-min catchment ─────
from IPython.display import HTML

scenario = generate_recommendations(
    df              = df,
    anchor_region   = "Kyoto",
    group_size      = 35,
    tourist_type    = "international",
    max_commute_mins= 180,
)

DISPLAY = [
    "region", "tfidf_score", "knn_profile_score",
    "congestion_flag", "dynamic_penalty",
    "transport_mode", "final_match_score",
]

if not scenario.empty:
    styled = (
        scenario[DISPLAY].style
        .format({
            "tfidf_score":       "{:.2f}",
            "knn_profile_score": "{:.2f}",
            "final_match_score": "{:.2f}",
        })
        .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
        .background_gradient(subset=["tfidf_score"],       cmap="Blues")
        .background_gradient(subset=["knn_profile_score"], cmap="PuBuGn")
        .map(
            lambda v: "color: crimson; font-weight: bold"
                if v == "Severely overtouristed"
            else ("color: darkorange; font-weight: bold"
                if v == "Approaching overtourism" else ""),
            subset=["congestion_flag"]
        )
        .set_caption(
            "Phase 2F+3 | Kyoto | Coach Tour (n=35) | International | 180 min"
        )
        .set_table_styles([{"selector": "", "props": [("width", "100%")]}])
    )
    display(HTML(
        '<div style="overflow-x:auto; max-width:100%">'
        + styled.to_html()
        + "</div>"
    ))
else:
    print("No recommendations within catchment.")


,region,tfidf_score,knn_profile_score,congestion_flag,dynamic_penalty,transport_mode,final_match_score
0,Koyasan,14.56,83.24,Undertouristed,25,Private Coach,-3.44
1,Nara,32.45,99.06,Severely overtouristed,10,Private Coach,-10.46


---
## Step 5 — Interactive Dashboard

The ipywidgets panel below exposes all three user parameters:
- **Anchor** dropdown (all 30 locations)
- **Group Size** slider (1–50)
- **Tourist Type** dropdown (shifts attractiveness weights)

The dashboard re-runs the full two-stage pipeline on every parameter change.

In [6]:
# ── Interactive Dashboard (Phase 2F + Dynamic Spatial) ────────────────────────
out = widgets.Output()

anchor_dropdown = widgets.Dropdown(
    options=sorted(df["region"].unique().tolist()),
    value="Kyoto",
    description="Anchor:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

group_slider = widgets.IntSlider(
    value=10, min=1, max=50, step=1,
    description="Group Size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

type_dropdown = widgets.Dropdown(
    options=["international", "native"],
    value="international",
    description="Tourist Type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

commute_slider = widgets.IntSlider(
    value=180, min=30, max=300, step=15,
    description="Max Commute (mins):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)


def render_recommendations(anchor_region, group_size, tourist_type, max_commute):
    results = generate_recommendations(
        df=df, anchor_region=anchor_region,
        group_size=group_size, tourist_type=tourist_type,
        max_commute_mins=max_commute,
    )

    DISPLAY = [
        "region", "tfidf_score", "knn_profile_score",
        "congestion_flag", "dynamic_penalty",
        "transport_mode", "final_match_score",
    ]

    with out:
        clear_output(wait=True)
        if results.empty:
            print("No thematically relevant orbits found within the catchment area.")
            return

        styled = (
            results[DISPLAY].style
            .format({
                "tfidf_score":       "{:.2f}",
                "knn_profile_score": "{:.2f}",
                "final_match_score": "{:.2f}",
            })
            .background_gradient(subset=["final_match_score"], cmap="RdYlGn")
            .background_gradient(subset=["tfidf_score"],       cmap="Blues")
            .background_gradient(subset=["knn_profile_score"], cmap="PuBuGn")
            .map(
                lambda v: "color: crimson; font-weight: bold"
                    if v == "Severely overtouristed"
                else ("color: darkorange; font-weight: bold"
                    if v == "Approaching overtourism" else ""),
                subset=["congestion_flag"],
            )
            .set_caption(
                f"DSS Recommendations | Anchor: {anchor_region} | "
                f"Group: {group_size} | Type: {tourist_type.capitalize()} | "
                f"Catchment: {max_commute} min"
            )
            .set_table_styles([{"selector": "", "props": [("width", "100%")]}])
        )
        display(HTML(
            '<div style="overflow-x:auto; max-width:100%">'
            + styled.to_html()
            + "</div>"
        ))


def _on_change(_):
    render_recommendations(
        anchor_dropdown.value,
        group_slider.value,
        type_dropdown.value,
        commute_slider.value,
    )


anchor_dropdown.observe(_on_change, names="value")
group_slider.observe(_on_change,    names="value")
type_dropdown.observe(_on_change,   names="value")
commute_slider.observe(_on_change,  names="value")

controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>DSS Dashboard — Dynamic Spatial Layer</h3>"),
    widgets.HBox([anchor_dropdown, type_dropdown]),
    group_slider,
    commute_slider,
    widgets.HTML(
        "<small style='color:grey'>Stage 1 (TF-IDF) filters thematically irrelevant orbits. "
        "Stage 2 (KNN) re-ranks by quality profile. "
        "Dynamic Spatial layer calculates transport mode from group size and distance. "
        "Group ≥ 15 → Private Coach + friction. FIT → Local/Express/Shinkansen.</small>"
    ),
])

display(controls, out)

# Trigger initial render
render_recommendations(
    anchor_dropdown.value,
    group_slider.value,
    type_dropdown.value,
    commute_slider.value,
)


Output()

---
## Architectural Justification — Why TF-IDF + KNN Supersedes Jaccard/Dice

### The Problem with Set-Theoretic Coefficients

Phase 2 Draft used **Jaccard similarity** to gate thematic relevance:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

While mathematically sound, Jaccard (and its cousin Dice) suffer from two critical failures when applied to rich tag taxonomies:

---

#### Failure 1: The Linear Tax on Tag-Set Size

Jaccard's denominator is $|A \cup B|$. As tag sets grow from 3–4 tags to 10–15 tags, the union inflates rapidly, **diluting even strong overlaps**. Two destinations sharing 6 out of 15 tags yield $J = 6 / 24 = 0.25$ — the same score as sharing 1 out of 4 tags ($J = 1 / 4 = 0.25$). The coefficient cannot distinguish between a genuinely strong but partial match and a trivial match. Dice partially mitigates this with $2|A \cap B|$ in the numerator, but it still treats every shared tag as equally important, which leads directly to Failure 2.

#### Failure 2: All Tags Are Treated Equally

In a 30-location dataset, `historic` appears in 18 locations and `geisha` in 2. Jaccard/Dice weight both identically. A destination sharing `historic` and `nature` with Kyoto scores the same overlap as one sharing `geisha` and `matcha` — despite the latter pair being *far more thematically distinctive*. This produces **generically correct but practically useless** rankings.

---

### Why TF-IDF Solves Both Failures

TF-IDF's **inverse document frequency** term naturally down-weights tags that appear across many locations:

$$\text{IDF}(t) = \log\frac{N}{\text{df}(t)}$$

where $N$ is the total number of locations and $\text{df}(t)$ is the count of locations containing tag $t$.

- `historic` (df=18): $\text{IDF} = \log(30/18) = 0.22$ — near-zero weight
- `geisha` (df=2): $\text{IDF} = \log(30/2) = 1.18$ — high weight

This means the cosine similarity between Kyoto and Kanazawa (sharing `geisha`, `matcha`, `crafts`, `cherry_blossom`, `autumn_leaves`) scores much higher than between Kyoto and a destination sharing only `historic` and `temple` — because the shared tags are **distinctive to that thematic corridor**, not just commonly appearing.

Cosine similarity also normalises by vector magnitude, making it **independent of tag-set size**. A 12-tag location and a 15-tag location are compared purely on directional alignment in TF-IDF space, not penalised by their difference in cardinality.

---

### Why KNN Adds Value as a Re-Ranker

TF-IDF captures *thematic* similarity but is blind to *quality* similarity. Two destinations may share the same tags but differ dramatically in infrastructure, cleanliness, or experiential richness. A visitor drawn to Kyoto's high cultural score and moderate infrastructure should not be sent to a destination with matching tags but collapsing infrastructure.

KNN on normalised numerical features (`cultural_score`, `experiential_score`, `cleanliness_score`, `infrastructure_score`) evaluates **holistic destination quality**. By operating as a Stage 2 re-ranker — scoring only the Stage 1 survivors — KNN avoids the noise of comparing thematically irrelevant destinations and focuses the quality signal on genuinely viable alternatives.

---

### Summary: Phase 2 Evolution

| Phase | Method | Weakness |
|---|---|---|
| Phase 2 Draft | Jaccard on 3–4 tags | Tag-size dilution, no IDF weighting |
| Phase 2F (Final) | TF-IDF + KNN pipeline | ✅ IDF weights rare tags, KNN evaluates quality |

The two-stage architecture cleanly separates *"Is this destination thematically relevant?"* (TF-IDF) from *"Is this destination qualitatively similar?"* (KNN), producing rankings that are both thematically coherent and practically useful for diversion decisions.

---
## Phase 3: Spatial Graph Layer — Dynamic Transport Profile

### The Netflix Problem

A purely thematic engine commits the **Netflix Problem**: recommending Sapporo to a Kyoto visitor — a perfect thematic match but geographically impossible within an itinerary window. The Spatial Graph Layer eliminates this class of failure.

### Tortuosity Multiplier (1.4×)

The Haversine formula returns the **great-circle distance** — the shortest path through the Earth's curvature. Real roads and rail tracks are not straight lines. Japan's terrain (mountains, coastline, urban density) means actual route distances average **1.3–1.5×** the Haversine distance. We use 1.4× as a conservative empirical estimate:

$$d_{\text{route}} = d_{\text{haversine}} \times 1.4$$

### Group Size → Transport Profile

Group size logically dictates the available transport mode:

| Group Size | Transport Mode | Assumed Speed | Rationale |
|---|---|---|---|
| ≥ 15 | Private Coach | 65 km/h | Coach hire is standard for groups; speed reflects rest stops, boarding time, and expressway limits |
| < 15, < 50 km | Local Transit | 40 km/h | Local trains and buses serve short-distance connections |
| < 15, 50–150 km | Limited Express | 80 km/h | JR Limited Express covers mid-range inter-city routes |
| < 15, > 150 km | Shinkansen | 150 km/h | Bullet train for long-distance; 150 accounts for station dwell time |

### Softened Penalty (Max 15 Points)

The previous implementation applied a penalty of up to 30 points for a 120-minute commute, which **catastrophically collapsed** scores for mid-range destinations. The softened formula scales gently to a **maximum of 15 points** at the catchment boundary:

$$\text{penalty} = \frac{\text{commute\_minutes}}{\text{max\_commute\_mins}} \times 15$$

For coach groups (≥ 15), a **1.5× friction multiplier** reflects the logistical overhead of parking, boarding, rest stops, and contractual time constraints.

### Three-Layer DSS Architecture

| Layer | Filter Type | Question Answered |
|---|---|---|
| **Stage 1: TF-IDF** | Thematic hard filter | *"Does this destination match why the tourist chose the anchor?"* |
| **Stage 2: KNN** | Quality re-ranker | *"Does this destination's quality profile mirror the anchor's?"* |
| **Phase 3: Spatial** | Dynamic transport + penalty | *"Can this group physically reach there?"* |